# AMEX Enterprise Credit Risk Platform
## Notebook 02 — Data Engineering
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Data Preparation**. Sprint 1, Notebook 2 of 18. Depends on Notebook 01 (reads `artifacts/project_config.json`) -- run Notebook 01 first if you have not already.

**What this notebook does:** ingests the two real, official AMEX raw files (`train_data.csv`, 16.4GB / 5,531,451 statement rows, and `test_data.csv`, 33.8GB / 11,363,762 statement rows) with Polars' lazy, streaming engine, aggregates each down to one row per customer, attaches the real labels to the training population, and splits **only the labeled population** into an internal train/validation set at a stratified 80/20 ratio. `test_data.csv` is aggregated into its own feature store but is never split and never touched by any label -- it has none, and is reserved solely for generating a final submission once a model exists (Notebook 05 onward).

**Zero-fabrication rule (same as Notebook 01):** every number below -- row counts, customer counts, wall-clock time, memory used, split sizes -- is computed live by this cell during this run. Nothing is carried over from any prior session.

**Run the single code cell below, once.** It is idempotent: every output file is written to the same fixed path and overwritten in place on every re-run -- re-running never creates a second, third, or renamed copy of anything. Depending on disk speed this cell can take a meaningful amount of wall-clock time (it streams ~50GB of raw CSV total) -- that is expected; progress banners print as each stage completes.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOK 01
# =============================================================================
import os
import sys
import json
import time
import gc
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebook 01")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"{CONFIG_PATH} not found.\n"
        "Fix: run 01_business_understanding.ipynb first -- it writes this shared "
        "config file, and every notebook from here on reads it instead of "
        "redefining PROJECT_ROOT, DATA_ROOT, RANDOM_SEED, and the pillar folder "
        "paths eighteen separate times."
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

DATA_ROOT = Path(PROJECT_CONFIG["data_root"])
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
WARP_THREAD_COUNT = (
    PROJECT_CONFIG.get("resource_limits", {}).get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)  # 95% of logical cores when Notebook 01's resource_limits block is present;
   # falls back to the raw core count on an older project_config.json.

TRAIN_CSV = DATA_ROOT / "train_data.csv"
TEST_CSV = DATA_ROOT / "test_data.csv"
TRAIN_LABELS_CSV = DATA_ROOT / "train_labels.csv"

for _p in (TRAIN_CSV, TEST_CSV, TRAIN_LABELS_CSV):
    if not _p.exists():
        raise FileNotFoundError(f"Required file not found: {_p}")

print(f"Loaded config from   : {CONFIG_PATH}")
print(f"PROJECT_ROOT          : {PROJECT_ROOT}")
print(f"DATA_ROOT              : {DATA_ROOT}")
print(f"RANDOM_SEED            : {RANDOM_SEED}")
print(f"Logical cores (from Notebook 01's live detection): {DETECTED_LOGICAL_CORES}")
print(f"train_data.csv         : {TRAIN_CSV.stat().st_size / 1e9:.2f} GB (confirmed present)")
print(f"test_data.csv          : {TEST_CSV.stat().st_size / 1e9:.2f} GB (confirmed present)")
print(f"train_labels.csv       : {TRAIN_LABELS_CSV.stat().st_size / 1e6:.2f} MB (confirmed present)")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

# --- WARP Section 6.4, Concurrency: Polars reads its thread pool size from the
#     POLARS_MAX_THREADS environment variable, and only at import time -- so
#     this MUST be set before "import polars" runs, not after. Using the
#     logical core count Notebook 01 detected live on this machine (rather
#     than a hardcoded "16") means this notebook auto-adapts to whatever
#     machine actually runs it. ---
os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    from sklearn.model_selection import train_test_split
except ImportError:
    missing.append("scikit-learn")
try:
    import psutil
    _HAVE_PSUTIL = True
except ImportError:
    _HAVE_PSUTIL = False
    logger.warning("psutil not installed -- peak memory readings will show as null. Optional: pip install psutil.")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (WARP 6.4, Concurrency)")
logger.info(f"Polars version: {pl.__version__}")
print("\n\u2705 Section 2 complete -- polars, scikit-learn imported; thread pool set before polars import.")


# =============================================================================
# SECTION 3: LIVE DATASET SCHEMA DETECTION
# =============================================================================
_section("SECTION 3: Live Dataset Schema Detection")

# --- Known from the official AMEX data dictionary and confirmed against the
#     real column values earlier in this project: these 11 columns are
#     categorical (D_117 in particular has a VALID category code of -1 -- it
#     is not a missing-value sentinel and must not be treated as one). Every
#     other feature column is numeric. This list is dataset structure, not a
#     computed result, so it is a documented constant rather than something
#     re-derived from a CSV header (dtype is not recoverable from a header
#     row alone). ---
CATEGORICAL = ["B_30", "B_38", "D_63", "D_64", "D_66", "D_68",
               "D_114", "D_116", "D_117", "D_120", "D_126"]

with open(TRAIN_CSV, "r", encoding="utf-8") as f:
    train_header = f.readline().strip().split(",")
with open(TEST_CSV, "r", encoding="utf-8") as f:
    test_header = f.readline().strip().split(",")

if train_header != test_header:
    raise ValueError(
        "train_data.csv and test_data.csv do not share the same column schema -- "
        f"train has {len(train_header)} columns, test has {len(test_header)}. "
        "Fix: confirm both files are the unmodified official AMEX CSVs before proceeding."
    )

feature_columns = [c for c in train_header if c not in ("customer_ID", "S_2")]
categorical_cols = [c for c in feature_columns if c in CATEGORICAL]
numeric_cols = [c for c in feature_columns if c not in CATEGORICAL]

print(f"Live-read header from {TRAIN_CSV.name} and {TEST_CSV.name} -- schemas match.")
print(f"Total raw feature columns : {len(feature_columns)}")
print(f"Categorical columns        : {len(categorical_cols)} -> {categorical_cols}")
print(f"Numeric columns            : {len(numeric_cols)}")
print(f"\nPer-customer aggregation will produce "
      f"{len(numeric_cols) * 5 + len(categorical_cols) * 2 + 2} engineered columns "
      f"(5 stats [mean/std/min/max/last] per numeric column, 2 stats [last/nunique] "
      f"per categorical column, plus statement_count and tenure_days).")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: REUSABLE CUSTOMER-LEVEL AGGREGATION PIPELINE (POLARS, STREAMING)
# =============================================================================
_section("SECTION 4: Reusable Customer-Level Aggregation Pipeline")

# --- One function, used for both train_data.csv and test_data.csv -- WARP
#     Section 6.4 (this project's own no-duplicated-logic principle): the
#     aggregation rules must be identical for both populations, so they are
#     defined exactly once. ---


def build_customer_feature_store(csv_path: Path, numeric_cols: list, categorical_cols: list) -> "pl.DataFrame":
    """Streams csv_path (customer_ID, S_2, + raw feature columns) and returns
    one aggregated row per customer_ID.

    WARP alignment:
      - Zero-Copy/Direct I/O   : pl.scan_csv + engine="streaming" reads and
                                  aggregates out-of-core -- the raw file is
                                  never fully materialized in RAM.
      - Cache-Friendly Layout  : schema_overrides fixes every column's dtype
                                  at scan time (Float32 for numeric, Utf8 for
                                  categorical) instead of inferring then
                                  re-casting in a second pass.
      - Concurrency            : Polars' streaming engine auto-parallelizes
                                  across POLARS_MAX_THREADS, set in Section 2.
      - Mechanical Sympathy    : explicit sort(["customer_ID","S_2"]) before
                                  the group_by guarantees `.last()` returns
                                  the chronologically last statement's value
                                  regardless of the order chunks are read in
                                  -- correctness first, even though it costs
                                  an out-of-core sort on a multi-GB file.

    Correctness note (raw-value inf cleaning, before any aggregation): the
    real AMEX raw files contain literal "inf"/"-inf" tokens in some numeric
    columns (confirmed on this project's own real hardware -- this is what
    originally crashed `.is_infinite()` on a string-inferred column and led
    to this notebook's explicit schema_overrides above). Once explicitly
    typed as Float32, those tokens parse to genuine floating-point infinity
    -- and infinity flowing uncleaned into `.std()` / `.mean()` here produces
    NaN (inf - inf, inf * 0, etc. in the underlying variance computation),
    NOT a Polars null. A later `.is_infinite()`-only cleaning pass (Notebook
    05 Section 5) does not catch that NaN, since is_infinite() is false for
    NaN by definition -- so an inf token on a single raw statement value
    would otherwise silently reappear as NaN in this customer's aggregated
    _std/_mean and, downstream, in Notebook 04's trend_slope (which is also
    a variance/covariance identity), and crash model training much later
    with no obvious link back to its true source. Cleaning inf -> null on
    the raw numeric columns immediately after scanning, before any
    aggregation touches them, removes this failure mode at its origin.
    """
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in categorical_cols:
        schema_overrides[c] = pl.Utf8
    for c in numeric_cols:
        schema_overrides[c] = pl.Float32

    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in numeric_cols
    ]

    agg_exprs = []
    for c in numeric_cols:
        agg_exprs += [
            pl.col(c).mean().alias(f"{c}_mean"),
            pl.col(c).std().alias(f"{c}_std"),
            pl.col(c).min().alias(f"{c}_min"),
            pl.col(c).max().alias(f"{c}_max"),
            pl.col(c).last().alias(f"{c}_last"),
        ]
    for c in categorical_cols:
        agg_exprs += [
            pl.col(c).last().alias(f"{c}_last"),
            # NOTE: Polars' n_unique() counts null as its own distinct value,
            # unlike pandas' nunique() (dropna=True by default) -- calling it
            # on the raw column would silently inflate every _nunique count
            # by 1 whenever a customer has even one missing statement for
            # that field, which is common in this dataset. drop_nulls()
            # first makes this match the standard "distinct non-null
            # categories seen" interpretation.
            pl.col(c).drop_nulls().n_unique().alias(f"{c}_nunique"),
        ]
    agg_exprs += [
        pl.len().alias("statement_count"),
        (pl.col("S_2").max() - pl.col("S_2").min()).dt.total_days().alias("tenure_days"),
    ]

    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .group_by("customer_ID", maintain_order=False)
        .agg(agg_exprs)
        .sort("customer_ID")
    )
    return lf.collect(engine="streaming")


print("build_customer_feature_store() defined -- shared by Sections 5 and 8 below.")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: AGGREGATE train_data.csv -> TRAIN CUSTOMER FEATURE STORE
# =============================================================================
_section("SECTION 5: Aggregate train_data.csv -> Train Customer Feature Store")

print(f"Streaming {TRAIN_CSV.name} ({TRAIN_CSV.stat().st_size / 1e9:.2f} GB, "
      f"5,531,451 raw rows expected) -- this may take a while depending on disk speed.")

_t0 = time.time()
train_features = build_customer_feature_store(TRAIN_CSV, numeric_cols, categorical_cols)
_train_agg_seconds = time.time() - _t0

_train_peak_mem_gb = None
if _HAVE_PSUTIL:
    _train_peak_mem_gb = round(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3), 2)

print(f"Aggregated {TRAIN_CSV.name} -> {train_features.shape[0]:,} customers x "
      f"{train_features.shape[1]} columns in {_train_agg_seconds:.1f}s "
      f"({TRAIN_CSV.stat().st_size / 1e6 / max(_train_agg_seconds, 0.001):.1f} MB/s effective throughput)")
if _train_peak_mem_gb:
    print(f"Process RSS after this stage: {_train_peak_mem_gb} GB")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: ATTACH LABELS & VALIDATE THE JOIN
# =============================================================================
_section("SECTION 6: Attach Labels & Validate the Join")

labels_df = pl.read_csv(str(TRAIN_LABELS_CSV), schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
print(f"Live-read {TRAIN_LABELS_CSV.name}: {labels_df.shape[0]:,} labeled customers")

train_labeled = train_features.join(labels_df, on="customer_ID", how="inner")

# --- Hard validation, not an assumption: every aggregated customer must have
#     matched exactly one label, and no customers may have been silently
#     dropped or duplicated by the join. ---
if train_labeled.shape[0] != train_features.shape[0]:
    raise RuntimeError(
        f"Join mismatch: {train_features.shape[0]:,} aggregated customers but "
        f"{train_labeled.shape[0]:,} rows after joining train_labels.csv. "
        "Fix: investigate customer_ID formatting differences between train_data.csv "
        "and train_labels.csv before proceeding -- do not silently continue with a "
        "partial join."
    )
if train_labeled.shape[0] != labels_df.shape[0]:
    raise RuntimeError(
        f"Row count mismatch after join: expected {labels_df.shape[0]:,} (from "
        f"train_labels.csv) but got {train_labeled.shape[0]:,}. Fix: check for "
        "duplicate customer_ID values in either source file."
    )

_live_default_rate = train_labeled["target"].mean()
print(f"Join verified: {train_labeled.shape[0]:,} customers, 1:1 match, no drops, no duplicates.")
print(f"Live-computed default rate on the joined labeled population: {_live_default_rate:.4%}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: INTERNAL TRAIN / VALIDATION SPLIT (80/20, STRATIFIED, CUSTOMER-LEVEL)
# =============================================================================
_section("SECTION 7: Internal Train / Validation Split")

# --- Named "train_split" / "test_split" per the requested terminology, but
#     documented clearly: this "test_split" is the internal validation
#     hold-out carved out of the LABELED population -- it is NOT the official
#     Kaggle test_data.csv (which has no label and is never split; see
#     Section 8). Split ratio matches the ratio already used in this
#     project's real, verified baseline run (test_size=0.20), so results
#     stay comparable. Splitting happens on customer_ID -- each customer's
#     single aggregated row goes entirely to one side, so there is no
#     leakage. ---
SPLIT_RATIO_TEST_SIZE = 0.20

customer_ids = train_labeled["customer_ID"].to_numpy()
targets = train_labeled["target"].to_numpy()

train_ids, val_ids = train_test_split(
    customer_ids, test_size=SPLIT_RATIO_TEST_SIZE, random_state=RANDOM_SEED, stratify=targets
)
train_ids_set = set(train_ids.tolist())
val_ids_set = set(val_ids.tolist())

train_split_df = train_labeled.filter(pl.col("customer_ID").is_in(train_ids_set))
test_split_df = train_labeled.filter(pl.col("customer_ID").is_in(val_ids_set))

if train_split_df.shape[0] + test_split_df.shape[0] != train_labeled.shape[0]:
    raise RuntimeError(
        "Split row-count mismatch: train_split + test_split does not equal the full "
        "labeled population. Fix: investigate before writing any output files."
    )

print(f"Split ratio            : {1 - SPLIT_RATIO_TEST_SIZE:.0%} / {SPLIT_RATIO_TEST_SIZE:.0%}, stratified by target")
print(f"train_split (internal)  : {train_split_df.shape[0]:,} customers, "
      f"default rate {train_split_df['target'].mean():.4%}")
print(f"test_split (internal)   : {test_split_df.shape[0]:,} customers, "
      f"default rate {test_split_df['target'].mean():.4%}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: AGGREGATE test_data.csv -> TEST CUSTOMER FEATURE STORE (NO LABELS, NO SPLIT)
# =============================================================================
_section("SECTION 8: Aggregate test_data.csv -> Test Customer Feature Store")

print(f"Streaming {TEST_CSV.name} ({TEST_CSV.stat().st_size / 1e9:.2f} GB, "
      f"11,363,762 raw rows expected) -- this is the larger file and will take longer.")

_t0 = time.time()
test_features = build_customer_feature_store(TEST_CSV, numeric_cols, categorical_cols)
_test_agg_seconds = time.time() - _t0

_test_peak_mem_gb = None
if _HAVE_PSUTIL:
    _test_peak_mem_gb = round(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3), 2)

print(f"Aggregated {TEST_CSV.name} -> {test_features.shape[0]:,} customers x "
      f"{test_features.shape[1]} columns in {_test_agg_seconds:.1f}s "
      f"({TEST_CSV.stat().st_size / 1e6 / max(_test_agg_seconds, 0.001):.1f} MB/s effective throughput)")
if _test_peak_mem_gb:
    print(f"Process RSS after this stage: {_test_peak_mem_gb} GB")
print("\nNote: this feature store has no target column -- it is not used for training "
      "or validation anywhere in this platform, only for final scoring once a model exists.")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: WRITE ALL OUTPUTS (FIXED PATHS -- OVERWRITE IN PLACE, NEVER DUPLICATE)
# =============================================================================
_section("SECTION 9: Write All Outputs")

DATA_ENG_DIR = PILLAR_DIRS["data_engineering"]

train_full_path = DATA_ENG_DIR / "train_full_features.parquet"
train_split_path = DATA_ENG_DIR / "train_split.csv"
test_split_path = DATA_ENG_DIR / "test_split.csv"
test_features_path = DATA_ENG_DIR / "test_features.parquet"

train_labeled.write_parquet(train_full_path)
logger.info(f"Wrote {train_full_path} ({train_full_path.stat().st_size / 1e9:.2f} GB)")

train_split_df.write_csv(train_split_path)
logger.info(f"Wrote {train_split_path} ({train_split_path.stat().st_size / 1e9:.2f} GB)")

test_split_df.write_csv(test_split_path)
logger.info(f"Wrote {test_split_path} ({test_split_path.stat().st_size / 1e9:.2f} GB)")

test_features.write_parquet(test_features_path)
logger.info(f"Wrote {test_features_path} ({test_features_path.stat().st_size / 1e9:.2f} GB)")

print(f"\u2705 {train_full_path.name:<28} -- full labeled population ({train_labeled.shape[0]:,} customers), efficient primary format")
print(f"\u2705 {train_split_path.name:<28} -- internal train split ({train_split_df.shape[0]:,} customers), csv as requested")
print(f"\u2705 {test_split_path.name:<28} -- internal validation split ({test_split_df.shape[0]:,} customers), csv as requested")
print(f"\u2705 {test_features_path.name:<28} -- official Kaggle test population ({test_features.shape[0]:,} customers), unlabeled, parquet")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 9B: VERIFY OUTPUTS
# =============================================================================
_section("SECTION 9B: Verify Outputs Were Written Correctly")

_expected_files = [train_full_path, train_split_path, test_split_path, test_features_path]
all_ok = True
for fp in _expected_files:
    if fp.exists() and fp.stat().st_size > 0:
        print(f"\u2705 {fp.name:<28} {fp.stat().st_size / 1e6:>10,.1f} MB")
    else:
        all_ok = False
        print(f"\u274c MISSING OR EMPTY: {fp}")

if not all_ok:
    raise RuntimeError("One or more Notebook 02 output files failed to write. See \u274c lines above.")

print("\nAll Notebook 02 outputs verified present and non-empty.")
print("\n\u2705 Section 9B complete.")


# =============================================================================
# SECTION 10: WRITE NOTEBOOK 02 SUMMARY ARTIFACT (for Notebook 17's rollup)
# =============================================================================
_section("SECTION 10: Write Notebook 02 Summary Artifact")

notebook_02_summary = {
    "notebook": "02_data_engineering",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "engine": f"Polars {pl.__version__}, engine='streaming', {os.environ['POLARS_MAX_THREADS']} threads",
    "train_data_csv": {
        "size_gb": round(TRAIN_CSV.stat().st_size / 1e9, 2),
        "aggregation_seconds": round(_train_agg_seconds, 1),
        "customers_aggregated": train_features.shape[0],
        "peak_process_rss_gb": _train_peak_mem_gb,
    },
    "test_data_csv": {
        "size_gb": round(TEST_CSV.stat().st_size / 1e9, 2),
        "aggregation_seconds": round(_test_agg_seconds, 1),
        "customers_aggregated": test_features.shape[0],
        "peak_process_rss_gb": _test_peak_mem_gb,
    },
    "join_validation": {
        "labeled_customers": train_labeled.shape[0],
        "live_default_rate": round(float(_live_default_rate), 6),
    },
    "internal_split": {
        "ratio": f"{1 - SPLIT_RATIO_TEST_SIZE:.0%}/{SPLIT_RATIO_TEST_SIZE:.0%}",
        "stratified_by": "target",
        "random_seed": RANDOM_SEED,
        "train_split_customers": train_split_df.shape[0],
        "test_split_customers": test_split_df.shape[0],
    },
    "output_files": {p.name: str(p) for p in _expected_files},
}

summary_path = ARTIFACTS_DIR / "notebook_02_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_02_summary, f, indent=2)

print(f"\u2705 Saved -> {summary_path} (Notebook 17 reads this file to build the rolled-up Data Engineering section)")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 11: Notebook 02 Complete -- Handoff to Notebook 03")

print("NOTEBOOK 02: DATA ENGINEERING -- COMPLETE")
print(f"  train_data.csv aggregated  : {train_features.shape[0]:,} customers in {_train_agg_seconds:.1f}s")
print(f"  test_data.csv aggregated   : {test_features.shape[0]:,} customers in {_test_agg_seconds:.1f}s")
print(f"  Internal split             : {train_split_df.shape[0]:,} train / {test_split_df.shape[0]:,} validation "
      f"({1 - SPLIT_RATIO_TEST_SIZE:.0%}/{SPLIT_RATIO_TEST_SIZE:.0%})")
print(f"  Files produced             : 4")
print(f"    - {train_full_path.name}")
print(f"    - {train_split_path.name}")
print(f"    - {test_split_path.name}")
print(f"    - {test_features_path.name}")
print(f"  Next notebook              : 03_data_validation_eda.ipynb (Data Validation & EDA, Sprint 1)")
print("\n\u2705 Ready to proceed.")
